# Riesgo y predicción cuantitativa: factores asociados al mal riesgo crediticio

**Curso:** MCC002 - Probabilidad y Estadística Computacional 

**Grupo 5:** 
- Armando Castro Chaupis 
- Henry Sánchez Alvarado 
- Alex Segura Núñez

**Pregunta principal:** ¿Qué factores explican la probabilidad de que un solicitante sea clasificado como mal riesgo crediticio?

**Preguntas secundarias:**

1. Proporción global de malos créditos y su IC 95 %.
2. ¿La tasa de mal crédito difiere según el propósito del préstamo?
3. ¿La duración y el monto del crédito difieren entre buenos y malos créditos?
4. ¿Qué variables están asociadas con mayor riesgo crediticio?
5. ¿Cómo cambia la clasificación al modificar el umbral de decisión?

## 1. Importación de librerias

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import statsmodels.api as sm
import statsmodels.formula.api as smf
import sklearn
from scipy import stats 

### Establecemos valor de semilla
SEED = 2026
np.random.seed(SEED)

### Estilo de gráficos para matplotlib
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 12


# 2. Carga de dataset

import os, io, zipfile, urllib.request

UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/144/statlog+german+credit+data.zip"
CANDIDATOS = ["data/german.data", "german.data"]

ruta_datos = next((p for p in CANDIDATOS if os.path.exists(p)), None)

if ruta_datos is None:
    # Descarga documentada desde la fuente oficial (solo si no existe copia local)
    os.makedirs("data", exist_ok=True)
    print("Descargando dataset desde UCI...")
    with urllib.request.urlopen(UCI_ZIP_URL) as r:
        zf = zipfile.ZipFile(io.BytesIO(r.read()))
        zf.extract("german.data", path="data")
    ruta_datos = "data/german.data"

# Nombres de columnas según german.doc (atributos 1..20 + clase)
columnas = [
    "estado_cuenta", "duracion", "historial_credito", "proposito", "monto",
    "ahorros", "empleo_actual", "tasa_cuota", "estatus_personal_sexo",
    "otros_deudores", "anios_residencia", "propiedad", "edad",
    "otros_planes_pago", "vivienda", "n_creditos_banco", "trabajo",
    "n_dependientes", "telefono", "trabajador_extranjero", "clase",
]

df = pd.read_csv(ruta_datos, sep=" ", header=None, names=columnas)
print(f"Archivo cargado: {ruta_datos}")
print(f"Dimensiones: {df.shape[0]} observaciones x {df.shape[1]} columnas")
df.head()

### 3.1 Diccionario de variables

Construido a partir del dataset. Las 13 variables cualitativas usan códigos `A**`. 
Abajo se documenta el significado de cada código y se crea un mapeo de etiquetas.

| # | Variable (nombre en el notebook) | Tipo | Descripción |
|---|---|---|---|
| 1 | `estado_cuenta` | Categórica ordinal | Estado de la cuenta corriente: A11 (< 0 DM), A12 (0–200 DM), A13 (≥ 200 DM), A14 (sin cuenta) |
| 2 | `duracion` | Numérica (meses) | Duración del crédito |
| 3 | `historial_credito` | Categórica | A30–A34: de "sin créditos/todo pagado" a "cuenta crítica/créditos en otros bancos" |
| 4 | `proposito` | Categórica nominal | A40–A410: auto nuevo/usado, mobiliario, radio/TV, electrodomésticos, reparaciones, educación, reentrenamiento, negocio, otros |
| 5 | `monto` | Numérica (DM) | Monto del crédito |
| 6 | `ahorros` | Categórica ordinal | A61–A65: nivel de ahorros/bonos (A65 = desconocido/sin cuenta) |
| 7 | `empleo_actual` | Categórica ordinal | A71–A75: antigüedad en el empleo actual |
| 8 | `tasa_cuota` | Numérica (1–4) | Cuota como % del ingreso disponible |
| 9 | `estatus_personal_sexo` | Categórica | A91–A94: estado civil y sexo (composición histórica del dataset) |
| 10 | `otros_deudores` | Categórica | A101 ninguno, A102 co-solicitante, A103 garante |
| 11 | `anios_residencia` | Numérica (1–4) | Años en la residencia actual |
| 12 | `propiedad` | Categórica | A121 inmueble … A124 sin propiedad conocida |
| 13 | `edad` | Numérica (años) | Edad del solicitante |
| 14 | `otros_planes_pago` | Categórica | A141 banco, A142 tiendas, A143 ninguno |
| 15 | `vivienda` | Categórica | A151 alquilada, A152 propia, A153 gratuita |
| 16 | `n_creditos_banco` | Numérica | N.º de créditos existentes en este banco |
| 17 | `trabajo` | Categórica ordinal | A171–A174: de no calificado/no residente a directivo/independiente |
| 18 | `n_dependientes` | Numérica | Personas a cargo |
| 19 | `telefono` | Binaria | A191 no, A192 sí (registrado) |
| 20 | `trabajador_extranjero` | Binaria | A201 sí, A202 no |
| 21 | `clase` | **Variable respuesta** | 1 = buen riesgo, 2 = mal riesgo |

In [ ]:
# Mapeos de códigos Axy -> etiquetas legibles (según german.doc)
mapa_proposito = {
    "A40": "Auto nuevo", "A41": "Auto usado", "A42": "Mobiliario/equipos",
    "A43": "Radio/TV", "A44": "Electrodomésticos", "A45": "Reparaciones",
    "A46": "Educación", "A48": "Reentrenamiento", "A49": "Negocio", "A410": "Otros",
}
mapa_estado_cuenta = {
    "A11": "< 0 DM", "A12": "0 – 200 DM", "A13": "≥ 200 DM", "A14": "Sin cuenta",
}
mapa_ahorros = {
    "A61": "< 100 DM", "A62": "100 – 500 DM", "A63": "500 – 1000 DM",
    "A64": "≥ 1000 DM", "A65": "Desconocido/sin cuenta",
}
mapa_historial = {
    "A30": "Sin créditos/todo pagado", "A31": "Pagados en este banco",
    "A32": "Al día hasta ahora", "A33": "Retrasos en el pasado",
    "A34": "Cuenta crítica/otros bancos",
}

df["proposito_lbl"] = df["proposito"].map(mapa_proposito)
df["estado_cuenta_lbl"] = df["estado_cuenta"].map(mapa_estado_cuenta)

# Verificación de que no quedaron códigos sin mapear
assert df["proposito_lbl"].notna().all(), "Hay códigos de propósito sin mapear"
assert df["estado_cuenta_lbl"].notna().all(), "Hay códigos de estado de cuenta sin mapear"

vars_numericas = ["duracion", "monto", "tasa_cuota", "anios_residencia",
                  "edad", "n_creditos_banco", "n_dependientes"]
vars_categoricas = [c for c in columnas if c not in vars_numericas + ["clase"]]
print(f"Variables numéricas ({len(vars_numericas)}): {vars_numericas}")
print(f"Variables categóricas ({len(vars_categoricas)}): {vars_categoricas}")

## Parte 2

## 9. Inferencia a través de la frecuencia
### 9.1 ¿La tasa de mal crédito difiere según el propósito?

Deberíamos esperar que para cada propósito exista la misma proporción de créditos de `riesgo` y `no-riesgo` (lo esperado, $E_i$) respecto a la proporción de toda la muestra. Sin embargo, en la práctica esto no siempre ocurre (lo observado, $O_i$). Para evaluar si la tasa de crédito difiere según el propósito, utilizaremos la prueba de **chi-cuadrado** $\chi^2$. Esta prueba se usa debido a la naturaleza categorica de las variables. 

$$
\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}
$$

Recordar que los grados de libertad de la distribucion se calculan $g_{dl}=(r-1)(c-1)$. Donde `r` es el número de propositos y `c` es la cantidad de clases. Para declarar que es válido la prueba de $\chi^2$, se debe cumplir los siguientes puntos:
- No debe existir frecuencias esperadas menor a 1.
- Como máximo, el 20% de las frecuencias esperadas deben ser menores a 5.

Despues de satisfacer esto, esto es posible hallar el valor `p` que nos permite aceptar o rechazar la **hipotesis de independencia** al 5\%. 


En caso se rechace la hipotesis, ¿que tan fuerte es esta dependencia entre variables? Para ello usarémos  **V de Cramér** como tamaño de efecto:
$$V = \sqrt{\frac{\chi^2}{n \cdot \min(r-1, c-1)}}$$

Y se puede interpretar de la siguiente forma:

 | V          | Interpretación    |
| ---------- | ----------------- |
| 0          | Sin relación      |
| 0.10       | Relación pequeña  |
| 0.30       | Relación moderada |
| 0.50 o más | Relación fuerte   |



In [ ]:
## Codigo tal cual

tabla = pd.crosstab(df["proposito_lbl"], df["mal_credito"])
chi2, p_chi, dof, esperadas = stats.chi2_contingency(tabla)
V = np.sqrt(chi2 / (n * (min(tabla.shape) - 1)))
n_esp_bajas = int((esperadas < 5).sum())

print("--- Prueba chi-cuadrado: clase x propósito (10 categorías) ---")
print(f"chi2 = {chi2:.2f}, gl = {dof}, p = {p_chi:.5f}, V de Cramér = {V:.3f}")
print(f"Celdas con frecuencia esperada < 5: {n_esp_bajas} de {tabla.size}")

In [ ]:
## Permite ver que tipo de variable tiene baja frecuencia esperada.
esp_df = pd.DataFrame(
    esperadas,
    index=tabla.index,
    columns=tabla.columns
)
 
for fila in esp_df.index:
    for col in esp_df.columns:
        if esp_df.loc[fila, col] < 5:
            print(f"{fila} - {col}: {esp_df.loc[fila, col]:.2f}")

A fin de mejorar la robustez de la estimación por la prueba del 

In [ ]:
frec = df["proposito_lbl"].value_counts()
raros = frec[frec < 30].index.tolist()
df["proposito_grp"] = df["proposito_lbl"].where(~df["proposito_lbl"].isin(raros),
                                                "Otros (agrupado)")
tabla_g = pd.crosstab(df["proposito_grp"], df["mal_credito"])
chi2_g, p_g, dof_g, esp_g = stats.chi2_contingency(tabla_g)
V_g = np.sqrt(chi2_g / (n * (min(tabla_g.shape) - 1)))
print(f"\n--- Robustez con categorías raras agrupadas ({tabla_g.shape[0]} categorías) ---")
print(f"Agrupados: {raros}")
print(f"chi2 = {chi2_g:.2f}, gl = {dof_g}, p = {p_g:.5f}, V de Cramér = {V_g:.3f}, "
      f"esperadas < 5: {int((esp_g < 5).sum())}")

De ambos procedimientos, se puede observar que la prueba de Chi-cuadrado muestra dependencia estadistica significativa entre el proposito y el riesgo del credito. Sin embargo, es importante resaltar que el tamaño de efecto `V` es pequeño, indicando que su relación es pequeña.

### 9.2 ¿Difieren la duración y el monto entre buenos y malos créditos?

Debido a que son variables numericas, realizar una prueba de Chi-cuadrado no serviria. 

Por lo que, para **duración**: aplicamos la prueba **t de Welch**. Esta prueba no asume varianzas iguales como la **t de Student**. Al igual que en la sección anterior, para estimar que tan fuerte es esta dependencia usamos la `d de Cohen`
